# Phase 1: OME-Zarr Open + DatasetSummary (Rust Backend)

This notebook validates `POST /dataset/open` against the Rust daemon using a real fixture dataset.

In [1]:
from __future__ import annotations

import os
import shutil
import sys
import tempfile
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'tests').exists():
            return candidate
    raise RuntimeError('could not locate repository root from notebook cwd')


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
if str(REPO_ROOT / 'tests') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'tests'))

import httpx

from lucida.client import LucidaClient
from conftest import create_sample_omezarr
from rust_daemon import start_rust_daemon


In [2]:
tmp_dir = Path(tempfile.mkdtemp(prefix='lucida-phase1-open-rust-'))
dataset_uri = create_sample_omezarr(str(tmp_dir / 'sample.zarr'))
daemon = start_rust_daemon(repo_root=REPO_ROOT, env=dict(os.environ))
client = LucidaClient(base_url=daemon.base_url, backend='rust')
print({'dataset_uri': dataset_uri, 'base_url': daemon.base_url})


{'dataset_uri': '/var/folders/hs/qw7ws1q52153c4c639t_p3600000gn/T/lucida-phase1-open-rust-hj_as9li/sample.zarr', 'base_url': 'http://127.0.0.1:58703'}


In [3]:
opened = client.open_dataset(dataset_uri)
payload = opened.model_dump(mode='json')
summary = payload['dataset_summary']

assert payload['schema_version'] == 1
assert summary['schema_version'] == 1
assert summary['dtype'] == 'uint16'
assert summary['shape'] == [1, 2, 4, 8, 10]
assert summary['uri'].startswith('file://')
assert len(summary['axes']) == 5
assert len(summary['multiscales']) >= 1
assert payload['warnings'] == []

{
    'dataset_id': summary['dataset_id'],
    'axes': [axis['name'] for axis in summary['axes']],
    'multiscale_names': [entry['name'] for entry in summary['multiscales']],
}


{'dataset_id': 'ds_126a81b50ff3cbfc',
 'axes': ['t', 'c', 'z', 'y', 'x'],
 'multiscale_names': ['primary']}

In [4]:
with httpx.Client(base_url=daemon.base_url, timeout=30.0) as http_client:
    bad = http_client.post('/dataset/open', json={'schema_version': 1, 'uri': str(tmp_dir / 'missing.zarr')})

assert bad.status_code >= 400
bad_payload = bad.json()
assert isinstance(bad_payload.get('code'), str)
assert bool(bad_payload['code'])
assert isinstance(bad_payload.get('message'), str)
assert 'details' in bad_payload
bad_payload


{'code': 'dataset_open_failed',
 'message': 'Failed to open dataset store.',
 'details': {'reason': 'Dataset path does not exist.',
  'uri': 'file:///var/folders/hs/qw7ws1q52153c4c639t_p3600000gn/T/lucida-phase1-open-rust-hj_as9li/missing.zarr'}}

In [5]:
if 'client' in globals():
    client.close()
if 'daemon' in globals():
    daemon.stop()
if 'tmp_dir' in globals():
    shutil.rmtree(tmp_dir, ignore_errors=True)
